## Generate styled HTML from your Markdown notes (with date-prefixed filenames)

This notebook converts one `.md` file (or a folder of `.md` files) into **self-contained HTML** with an embedded CSS theme.

### What it does
- Reads Markdown notes from your repo
- Converts Markdown → HTML (with fenced code blocks, tables, and syntax highlighting)
- Writes HTML to an output folder
- Prefixes output filenames with today’s date: `YYYY-MM-DD_<note-name>.html`

### Typical use
1. Set `INPUT_PATH` to a specific `.md` file, or a folder containing many `.md` files.
2. Run all cells.
3. Upload the generated `.html` files to GitHub (works great with GitHub Pages, or just for downloading/opening locally).


In [ ]:
# First run prefix_read_filenames_with_created_date.py
# Then run the below code to generate the final html

%run -i "prefix_read_filenames_with_created_date.py"

In [ ]:
from __future__ import annotations

import datetime as _dt
import html as _html
import os
import re
from pathlib import Path

# --- CONFIG ---
# Set to:
# - a specific Markdown file, e.g. Path("JET_2025_Groh_Competitive_Price_Discrimination.md")
# - OR a folder to convert all .md files inside, e.g. Path(".")
INPUT_PATH = Path("./read")

# Where generated HTML files will be written
OUTPUT_DIR = Path("html")

# Skip converting these filenames when converting a folder
SKIP_FILES = {"README.MD", "README.md", "test.md"}

# Date prefix source for output HTML filenames
# - "created": use the file *created* date (macOS: st_birthtime)
# - "modified": use last modified date
# - "today": use today's date
DATE_PREFIX_MODE = "created"

# If True, overwrites existing same-named output files
# (Recommended: keep False to avoid accidental clobbering.)
OVERWRITE = False

# If True, skip converting a note when ANY prior generated HTML exists for the same note name
# in OUTPUT_DIR (even if the date prefix differs).
SKIP_IF_ALREADY_GENERATED = True

# MathJax (renders LaTeX math like $...$, $$...$$, \( ... \), \[ ... \])
# Note: this loads MathJax from a CDN (great for GitHub Pages / online viewing).
ENABLE_MATHJAX = True
MATHJAX_SRC = "https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"

In [ ]:
import sys
import subprocess
import importlib.util


def _ensure_importable(pkg: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(pkg) is None:
        name = pip_name or pkg
        print(f"Installing {name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])


# Python-Markdown for conversion + Pygments for code highlighting
_ensure_importable("markdown", "markdown")
_ensure_importable("pygments", "pygments")

import markdown  # noqa: E402


In [ ]:
# A lightweight, GitHub-ish CSS theme (embedded so each HTML is self-contained)
CSS = r"""
:root {
  --bg: #ffffff;
  --fg: #24292f;
  --muted: #57606a;
  --border: #d0d7de;
  --code-bg: #f6f8fa;
  --link: #0969da;
}

html, body {
  background: var(--bg);
  color: var(--fg);
  margin: 0;
  padding: 0;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, "Apple Color Emoji", "Segoe UI Emoji";
  line-height: 1.6;
}

.container {
  max-width: 980px;
  margin: 0 auto;
  padding: 28px 20px 60px;
}

.header {
  border-bottom: 1px solid var(--border);
  padding-bottom: 14px;
  margin-bottom: 20px;
}

.header .title {
  margin: 0;
  font-size: 28px;
  line-height: 1.25;
}

.header .meta {
  margin-top: 6px;
  color: var(--muted);
  font-size: 13px;
}

.markdown-body h1, .markdown-body h2, .markdown-body h3, .markdown-body h4 {
  margin-top: 1.2em;
  margin-bottom: 0.5em;
  line-height: 1.25;
}

.markdown-body h1 { font-size: 2em; }
.markdown-body h2 { font-size: 1.5em; border-bottom: 1px solid var(--border); padding-bottom: 0.3em; }
.markdown-body h3 { font-size: 1.25em; }

.markdown-body p { margin: 0.7em 0; }

.markdown-body a { color: var(--link); text-decoration: none; }
.markdown-body a:hover { text-decoration: underline; }

.markdown-body code {
  background: var(--code-bg);
  padding: 0.15em 0.35em;
  border-radius: 6px;
  font-size: 0.95em;
}

.markdown-body pre {
  background: var(--code-bg);
  padding: 14px;
  border-radius: 10px;
  overflow: auto;
  border: 1px solid var(--border);
}

.markdown-body pre code {
  background: transparent;
  padding: 0;
}

.markdown-body blockquote {
  margin: 1em 0;
  padding: 0.6em 1em;
  color: var(--muted);
  border-left: 4px solid var(--border);
  background: #fbfcfd;
}

.markdown-body table {
  width: 100%;
  border-collapse: collapse;
  margin: 1em 0;
}

.markdown-body th, .markdown-body td {
  border: 1px solid var(--border);
  padding: 8px 10px;
}

.markdown-body th {
  background: #f6f8fa;
  text-align: left;
}

.footer {
  margin-top: 30px;
  padding-top: 14px;
  border-top: 1px solid var(--border);
  color: var(--muted);
  font-size: 12px;
}

/* Pygments default-ish tweaks for codehilite */
.codehilite .hll { background-color: #ffffcc }
.codehilite  { background: #f6f8fa; }
.codehilite .c { color: #6a737d } /* Comment */
.codehilite .k { color: #d73a49 } /* Keyword */
.codehilite .s { color: #032f62 } /* String */
.codehilite .n { color: #24292f } /* Name */
.codehilite .o { color: #d73a49 } /* Operator */
.codehilite .p { color: #24292f } /* Punctuation */
"""


In [ ]:
def _slugify_filename(stem: str) -> str:
    # Keep filenames GitHub/URL friendly
    s = stem.strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9._-]+", "", s)
    s = re.sub(r"_+", "_", s)
    return s or "note"


def _gather_markdown_files(input_path: Path) -> list[Path]:
    p = input_path.expanduser().resolve()
    if p.is_file():
        if p.suffix.lower() != ".md":
            raise ValueError(f"INPUT_PATH is a file but not .md: {p}")
        return [p]

    if not p.is_dir():
        raise ValueError(f"INPUT_PATH does not exist: {p}")

    files = sorted([x for x in p.iterdir() if x.is_file() and x.suffix.lower() == ".md" and x.name not in SKIP_FILES])
    return files


def _render_markdown_to_html(md_text: str) -> str:
    md = markdown.Markdown(
        extensions=[
            "fenced_code",
            "tables",
            "toc",
            "codehilite",
            "md_in_html",
        ],
        extension_configs={
            "codehilite": {
                "guess_lang": False,
                "use_pygments": True,
                "noclasses": False,
            }
        },
        output_format="html5",
    )
    return md.convert(md_text)


def _date_prefix_for_file(path: Path) -> str:
    mode = (DATE_PREFIX_MODE or "created").lower()
    if mode == "today":
        return _dt.date.today().isoformat()

    st = path.stat()

    if mode == "created":
        # macOS provides true creation time via st_birthtime
        ts = getattr(st, "st_birthtime", None)
        if ts is None:
            # Fallback: on some OSes, ctime is metadata-change time; mtime is safer for content recency.
            ts = st.st_mtime
    elif mode == "modified":
        ts = st.st_mtime
    else:
        raise ValueError(f"Unknown DATE_PREFIX_MODE: {DATE_PREFIX_MODE!r}")

    return _dt.date.fromtimestamp(ts).isoformat()


def _wrap_full_html(*, title: str, source_path: Path, html_body: str) -> str:
    # Keep <title> for browser tabs/search, but do NOT inject any visible header.
    safe_title = _html.escape(title)

    mathjax_block = ""
    if ENABLE_MATHJAX:
        # Configure common delimiters, including $...$ and $$...$$
        mathjax_block = f"""
  <script>
    window.MathJax = {{
      tex: {{
        inlineMath: [['$', '$'], ['\\\\(', '\\\\)']],
        displayMath: [['$$', '$$'], ['\\\\[', '\\\\]']],
        processEscapes: true,
        processEnvironments: true
      }},
      options: {{
        skipHtmlTags: ['script', 'noscript', 'style', 'textarea', 'pre', 'code']
      }}
    }};
  </script>
  <script defer src=\"{_html.escape(MATHJAX_SRC)}\"></script>
"""

    return f"""<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
  <title>{safe_title}</title>
  <style>{CSS}</style>{mathjax_block}
</head>
<body>
  <div class=\"container\">
    <div class=\"markdown-body\">
{html_body}
    </div>
  </div>
</body>
</html>
"""


In [ ]:
md_files = _gather_markdown_files(INPUT_PATH)
if not md_files:
    raise RuntimeError(f"No .md files found under: {INPUT_PATH.resolve()}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_DATE_IN_NAME_RE = re.compile(r"^(\d{4}-\d{2}-\d{2})[_\-\s]+(.+)$")
_GENERATED_RE = re.compile(r"^(\d{4}-\d{2}-\d{2})_(.+)\.html$")


def _normalized_base(name_without_ext: str) -> str:
    """Derive a stable base name from a note stem or HTML base.

    - Strips an existing leading date prefix (YYYY-MM-DD_...)
    - Applies the same slugification used for output filenames
    """

    m = _DATE_IN_NAME_RE.match(name_without_ext)
    core = m.group(2) if m else name_without_ext
    return _slugify_filename(core)


def _looks_date_prefixed(name_without_ext: str) -> bool:
    return _DATE_IN_NAME_RE.match(name_without_ext) is not None


# Build an index of already-generated outputs so we can skip regeneration.
# We treat files named like: YYYY-MM-DD_<base>.html as "generated".
existing_generated: dict[str, Path] = {}
for hp in OUTPUT_DIR.glob("*.html"):
    m = _GENERATED_RE.match(hp.name)
    if not m:
        continue

    base_raw = m.group(2)
    base_key = _normalized_base(base_raw)

    prev = existing_generated.get(base_key)
    if prev is None:
        existing_generated[base_key] = hp
        continue

    # Prefer a "clean" (non-double-dated) base when both exist.
    prev_m = _GENERATED_RE.match(prev.name)
    prev_base_raw = prev_m.group(2) if prev_m else prev.stem

    prev_double = _looks_date_prefixed(prev_base_raw)
    cand_double = _looks_date_prefixed(base_raw)

    if prev_double and not cand_double:
        existing_generated[base_key] = hp
    elif prev_double == cand_double:
        # As a stable tie-breaker, keep the shorter filename (usually cleaner).
        if len(hp.name) < len(prev.name):
            existing_generated[base_key] = hp

written: list[Path] = []
skipped_existing = 0

for md_path in md_files:
    base_key = _normalized_base(md_path.stem)

    if SKIP_IF_ALREADY_GENERATED and base_key in existing_generated:
        print(f"SKIP (already generated): {md_path.name} -> {existing_generated[base_key].name}")
        skipped_existing += 1
        continue

    md_text = md_path.read_text(encoding="utf-8")
    html_body = _render_markdown_to_html(md_text)

    date_prefix = _date_prefix_for_file(md_path)
    out_name = f"{date_prefix}_{base_key}.html"
    out_path = (OUTPUT_DIR / out_name).resolve()

    if out_path.exists() and not OVERWRITE:
        print(f"SKIP (exists): {out_path}")
        skipped_existing += 1
        continue

    full_html = _wrap_full_html(title=md_path.stem, source_path=md_path, html_body=html_body)
    out_path.write_text(full_html, encoding="utf-8")
    written.append(out_path)

print(f"Converted {len(written)} file(s) → {OUTPUT_DIR.resolve()}")
if skipped_existing:
    print(f"Skipped {skipped_existing} file(s) (already existed)")
for p in written[:10]:
    print("-", p.name)
if len(written) > 10:
    print(f"... and {len(written) - 10} more")